In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
myworkingdirectory = r"C:\Users\Sophie.Kaye\Desktop\NPS Crash Data"
os.chdir(myworkingdirectory)

In [3]:
# raw data
imars_crash = pd.read_excel("./IMARS/IMARS 2012 - 2021.xlsx", sheet_name = "Location")
imars_passenger = pd.read_excel("./IMARS/IMARS 2012 - 2021.xlsx", sheet_name = "Person")
imars_vehicle = pd.read_excel("./IMARS/IMARS 2012 - 2021.xlsx", sheet_name = "Vehicle")
imars_crash_details = pd.read_excel("./IMARS/IMARS 2012 - 2021.xlsx", sheet_name = "Classification")

# Determine Any Fill and Non-Unknown Fill Percentages

## Raw Reports

### Crash Table

In [4]:
imars_crash.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Linked_Address_Classification',
       'City_Town_Park_Location', 'State', 'County', 'Direction',
       'Linked_Street_Number', 'Linked_Common_Name', 'Street_Type',
       'Direction.1', 'NEAR_Distance_to_MI', 'NEAR_Direction_To',
       'NEAR_Direction', 'NEAR_route_street_road_name', 'NEAR_Road_Type',
       'NEAR_Direction.1', 'AT_Intersection_route_street_road_DIRECTION',
       'At_Intersecting_route_street_road_name', 'AT_Road_Type',
       'AT_Direction', 'Mile_Marker', 'Latitude', 'Longitude', 'Region',
       'State_Zone', 'Park', 'Site', 'Place', 'Point',
       'Road_Type_Classification', 'Linked_Address'],
      dtype='object')

In [18]:
crashcolumns = imars_crash.columns
regions = ["NCR","NER","SER","MWR","IMR","PWR","AKR"]

for region in regions:
    df = imars_crash.loc[imars_crash['Region']==region] # note that this is the raw region data from crash reports, which is not nearly as populated as our cleaned data derived from Park name and will therefore drop MANY reports that cannot be sorted into a region
    filled = []
    filledNotUnknownPerc = []
    for i in crashcolumns:
        notNull = df.loc[df[i].notnull()]
        # determine fill percentages (counting "unknowns")
        full = np.nan_to_num(round(notNull.shape[0]/df.shape[0]*100,0)).astype(int) 
        filled.append(full)

        # drop reports on "null island", which is the lat/long equivalent of "unknown" to determine percentage of remaining reports filled with REAL lat/long data
        if i=="Latitude":
            fullNotUnknown = notNull.loc[notNull[i] != 0] 
            fullNotUnknownPerc = np.nan_to_num(round(fullNotUnknown.shape[0]/df.shape[0]*100,0)).astype(int)
            filledNotUnknownPerc.append(fullNotUnknownPerc)
        elif i=="Longitude":
            fullNotUnknown = notNull.loc[notNull[i] != 0] 
            fullNotUnknownPerc = np.nan_to_num(round(fullNotUnknown.shape[0]/df.shape[0]*100,0)).astype(int)
            filledNotUnknownPerc.append(fullNotUnknownPerc)
        else:
            fullNotUnknownPerc = 'NA'
            filledNotUnknownPerc.append(fullNotUnknownPerc)

    Filled = {'Column':crashcolumns, 'Fill Percentage':filled, 'Non-Unknown Fill Percentage':filledNotUnknownPerc}
    Filled_CrashTable = pd.DataFrame(Filled)
    Filled_CrashTable.to_csv("IMARS Crash Table Fill Percentages_" + region + ".csv", index=False)

### Passenger Table

In [23]:
# add region column to passenger table for parsing data into 7 reports
# note that this is the raw region data from crash reports, which is not nearly as populated as our cleaned data derived from Park name and will therefore drop MANY reports that cannot be sorted into a region
imars_passenger_expanded = pd.merge(imars_passenger, imars_crash[['Region','IMARS_Record_No']], how ='left', on = 'IMARS_Record_No')
imars_passenger_expanded.to_csv("imars_passenger_regioncolumnadded.csv", index=False)

passengercolumns = imars_passenger_expanded.columns
regions = ["NCR","NER","SER","MWR","IMR","PWR","AKR"]

for region in regions:
    df = imars_passenger_expanded.loc[imars_passenger_expanded['Region']==region] 
    filled = []
    filledNotUnknownPerc = []
    for i in passengercolumns:
        notNull = df.loc[df[i].notnull()]
        # determine fill percentages (including "unknowns")
        full = np.nan_to_num(round(notNull.shape[0]/df.shape[0]*100,0)).astype(int)
        filled.append(full)

        # determine percent of reports where the crash reporter entered something OTHER THAN the unknown attributes listed 
            # (drop reports with these exact values, but keep reports where unknown was one of many REAL data attributes selected: e.g., "20. Avoiding an object on road; 24. Over-correcting/over-steering; 99. Unknown")
        fullNotUnknown = notNull.loc[notNull[i].isin(["99. Unknown", "06. Unknown if distracted", "07. Unknown if distracted"])==False] 
        fullNotUnknownPerc = np.nan_to_num(round(fullNotUnknown.shape[0]/df.shape[0]*100,0)).astype(int)
        filledNotUnknownPerc.append(fullNotUnknownPerc)
    
    Filled = {'Column':passengercolumns, 'Fill Percentage':filled, 'Non-Unknown Fill Percentage':filledNotUnknownPerc}
    Filled_PassengerTable = pd.DataFrame(Filled)
    Filled_PassengerTable.to_csv("IMARS Passenger Table Fill Percentages_" + region+ ".csv", index=False)

### Vehicle Table

In [31]:
# add region column to vehicle table for parsing data into 7 reports
# note that this is the raw region data from crash reports, which is not nearly as populated as our cleaned data derived from Park name and will therefore drop MANY reports that cannot be sorted into a region
imars_vehicle_expanded = pd.merge(imars_vehicle, imars_crash[['Region','IMARS_Record_No']], how ='left', on = 'IMARS_Record_No')
imars_vehicle_expanded.to_csv("imars_vehicle_regioncolumnadded.csv", index=False)

vehiclecolumns = imars_vehicle_expanded.columns
regions = ["NCR","NER","SER","MWR","IMR","PWR","AKR"]

for region in regions:
    df = imars_vehicle_expanded.loc[imars_vehicle_expanded['Region']==region] 
    filled = []
    filledNotUnknownPerc = []
    for i in vehiclecolumns:
        notNull = df.loc[df[i].notnull()]
        # determine fill percentages (including "unknowns")
        full = np.nan_to_num(round(notNull.shape[0]/df.shape[0]*100,0)).astype(int)
        filled.append(full)

        # determine percent of reports where the crash reporter entered something OTHER THAN the unknown attributes listed 
            # (drop reports with these exact values, but keep reports where unknown was one of multiple REAL data attributes selected: e.g., "10. Lights (head, signal, tail); 99. Unknown")
        fullNotUnknown = notNull.loc[notNull[i].isin(["99. Unknown", "04. Unknown", "06. Unknown", "Unknown"])==False] 
        fullNotUnknownPerc = np.nan_to_num(round(fullNotUnknown.shape[0]/df.shape[0]*100,0)).astype(int)
        filledNotUnknownPerc.append(fullNotUnknownPerc)

    Filled = {'Column':vehiclecolumns, 'Fill Percentage':filled, 'Non-Unknown Fill Percentage':filledNotUnknownPerc}
    Filled_VehicleTable = pd.DataFrame(Filled)
    Filled_VehicleTable.to_csv("IMARS Vehicle Table Fill Percentages_" + region + ".csv", index=False)

### Crash Details Table

In [32]:
# add region column to crash detail table for parsing data into 7 reports
# note that this is the raw region data from crash reports, which is not nearly as populated as our cleaned data derived from Park name and will therefore drop MANY reports that cannot be sorted into a region
imars_crash_details_expanded = pd.merge(imars_crash_details, imars_crash[['Region','IMARS_Record_No']], how ='left', on = 'IMARS_Record_No')
imars_crash_details_expanded.to_csv("imars_crash_details_regioncolumnadded.csv", index=False)

crashdeatcolumns = imars_crash_details_expanded.columns
regions = ["NCR","NER","SER","MWR","IMR","PWR","AKR"]

for region in regions:
    df = imars_crash_details_expanded.loc[imars_crash_details_expanded['Region']==region] 
    filled = []
    filledNotUnknownPerc = []
    for i in crashdeatcolumns:
        notNull = df.loc[df[i].notnull()]
        # determine fill percentages (including "unknowns")
        full = np.nan_to_num(round(notNull.shape[0]/df.shape[0]*100,0)).astype(int)
        filled.append(full)

        # determine percent of reports where the crash reporter entered something OTHER THAN the unknown attributes listed 
            # (drop reports with these exact values, but keep reports where unknown was one of multiple REAL data attributes selected: e.g., "02. Weather; 06. Animal(s) in roadway; 99. Unknown")
        fullNotUnknown = notNull.loc[notNull[i].isin(["99. Unknown", "Unknown"])==False] 
        fullNotUnknownPerc = np.nan_to_num(round(fullNotUnknown.shape[0]/df.shape[0]*100,0)).astype(int)
        filledNotUnknownPerc.append(fullNotUnknownPerc)
        
    Filled = {'Column':crashdeatcolumns, 'Fill Percentage':filled, 'Non-Unknown Fill Percentage':filledNotUnknownPerc}
    Filled_CrashDetailsTable = pd.DataFrame(Filled)
    Filled_CrashDetailsTable.to_csv("IMARS Crash Details Table Fill Percentages_" + region + ".csv", index=False)

## Cleaned/derived data used for analysis and webmap

### Crash Table

In [29]:
# table containing cleaned park, region, lat, and long columns (see IMARS Fill Percentages.ipynb)
imars_crash_replaced = pd.read_csv("imars_crash_replaced.csv")
imars_crash_replaced.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Linked_Address_Classification',
       'City_Town_Park_Location', 'State', 'County', 'Direction',
       'Linked_Street_Number', 'Linked_Common_Name', 'Street_Type',
       'Direction.1', 'NEAR_Distance_to_MI', 'NEAR_Direction_To',
       'NEAR_Direction', 'NEAR_route_street_road_name', 'NEAR_Road_Type',
       'NEAR_Direction.1', 'AT_Intersection_route_street_road_DIRECTION',
       'At_Intersecting_route_street_road_name', 'AT_Road_Type',
       'AT_Direction', 'Mile_Marker', 'Latitude', 'Longitude', 'RGN',
       'State_Zone', 'Park', 'Site', 'Place', 'Point',
       'Road_Type_Classification', 'Linked_Address'],
      dtype='object')

In [30]:
cleancrashcolumns = imars_crash_replaced.columns
regions = ["NCR","NER","SER","MWR","IMR","PWR","AKR"]

for region in regions:
    df = imars_crash_replaced.loc[imars_crash_replaced['RGN']==region] # note that this is the cleaned data derived from Park name and will therefore keep as many reports as possible for each region
    filled = []
    filledNotUnknownPerc = []
    for i in cleancrashcolumns:
        notNull = df.loc[df[i].notnull()]
        # determine fill percentages (counting "unknowns")
        full = np.nan_to_num(round(notNull.shape[0]/df.shape[0]*100,0)).astype(int) 
        filled.append(full)

       # drop reports on "null island", which is the lat/long equivalent of "unknown" to determine percentage of remaining reports filled with REAL lat/long data
        if i=="Latitude":
            fullNotUnknown = notNull.loc[notNull[i] != 0] 
            fullNotUnknownPerc = np.nan_to_num(round(fullNotUnknown.shape[0]/df.shape[0]*100,0)).astype(int)
            filledNotUnknownPerc.append(fullNotUnknownPerc)
        elif i=="Longitude":
            fullNotUnknown = notNull.loc[notNull[i] != 0] 
            fullNotUnknownPerc = np.nan_to_num(round(fullNotUnknown.shape[0]/df.shape[0]*100,0)).astype(int)
            filledNotUnknownPerc.append(fullNotUnknownPerc)
        else:
            fullNotUnknownPerc = 'NA'
            filledNotUnknownPerc.append(fullNotUnknownPerc)
            
    Filled = {'Column':cleancrashcolumns, 'Fill Percentage':filled, 'Non-Unknown Fill Percentage':filledNotUnknownPerc}
    Filled_CrashTable = pd.DataFrame(Filled)
    Filled_CrashTable.to_csv("Cleaned IMARS Crash Table Fill Percentages_" + region + ".csv", index=False)

### Crash Details Table

In [34]:
# table containing cleaned lighting column (see IMARS Fill Percentages.ipynb)
imars_crash_details_replaced = pd.read_csv("imars_crash_details_replaced.csv")
imars_crash_details_replaced.columns

Index(['IMARS_Record_No', 'Crash_Date_Time', 'Number_of_Vehicles_Involved',
       'Injury_or_Fatal_Crash', 'Investigated_at_Scene', 'Hit_and_Run',
       'Non_Motor_Vehicl_Property_Damage', 'Amount_of_Property_Damage',
       'First_Harmful_Event_Type', 'First_Harmful_Event',
       'Location_of_First_Harmful_Event', 'Weather', 'Roadway_Condition',
       'Lighting', 'School_Bus_related', 'AS_Road_Circumstance',
       'Environmental_Contributing_Circumstances', 'Work_Zone_Related',
       'Work_Zone_Workers_Present', 'Work_Zone_Location',
       'Law_Enforcement_Present_at_Work_Zone', 'Relation_to_Junction',
       'Type_of_Intersection', 'Manner_of_Collision'],
      dtype='object')

In [35]:
# add region column to crash details table for parsing data into 7 reports
# note that this is the cleaned region data derived from Park name and will therefore keep as many reports as possible for each region
imars_crash_details_expanded = pd.merge(imars_crash_details_replaced, imars_crash_replaced[['RGN','IMARS_Record_No']], how ='left', on = 'IMARS_Record_No')
imars_crash_details_expanded.to_csv("imars_crash_details_rgncolumnadded.csv", index=False)

crashdeatcolumns = imars_crash_details_expanded.columns
regions = ["NCR","NER","SER","MWR","IMR","PWR","AKR"]

for region in regions:
    df = imars_crash_details_expanded.loc[imars_crash_details_expanded['RGN']==region] 
    filled = []
    filledNotUnknownPerc = []
    for i in crashdeatcolumns:
        notNull = df.loc[df[i].notnull()]
        # determine fill percentages (including "unknowns")
        full = np.nan_to_num(round(notNull.shape[0]/df.shape[0]*100,0)).astype(int)
        filled.append(full)

        # determine percent of reports where the crash reporter entered something OTHER THAN the unknown attributes listed 
            # (drop reports with these exact values, but keep reports where unknown was one of multiple REAL data attributes selected: e.g., "02. Weather; 06. Animal(s) in roadway; 99. Unknown")
        fullNotUnknown = notNull.loc[notNull[i].isin(["99. Unknown", "Unknown"])==False] 
        fullNotUnknownPerc = np.nan_to_num(round(fullNotUnknown.shape[0]/df.shape[0]*100,0)).astype(int)
        filledNotUnknownPerc.append(fullNotUnknownPerc)
        
    Filled = {'Column':crashdeatcolumns, 'Fill Percentage':filled, 'Non-Unknown Fill Percentage':filledNotUnknownPerc}
    Filled_CrashDetailsTable = pd.DataFrame(Filled)
    Filled_CrashDetailsTable.to_csv("Cleaned IMARS Crash Details Table Fill Percentages_" + region + ".csv", index=False)

### Passenger Table

In [36]:
# add region column to passenger table for parsing data into 7 reports
# note that this is the cleaned region data derived from Park name and will therefore keep as many reports as possible for each region
imars_passenger_expanded = pd.merge(imars_passenger, imars_crash_replaced[['RGN','IMARS_Record_No']], how ='left', on = 'IMARS_Record_No')
imars_passenger_expanded.to_csv("imars_passenger_rgncolumnadded.csv", index=False)

passengercolumns = imars_passenger_expanded.columns
regions = ["NCR","NER","SER","MWR","IMR","PWR","AKR"]

for region in regions:
    df = imars_passenger_expanded.loc[imars_passenger_expanded['RGN']==region] 
    filled = []
    filledNotUnknownPerc = []
    for i in passengercolumns:
        notNull = df.loc[df[i].notnull()]
        # determine fill percentages (including "unknowns")
        full = np.nan_to_num(round(notNull.shape[0]/df.shape[0]*100,0)).astype(int)
        filled.append(full)

        # determine percent of reports where the crash reporter entered something OTHER THAN the unknown attributes listed 
            # (drop reports with these exact values, but keep reports where unknown was one of many REAL data attributes selected: e.g., "20. Avoiding an object on road; 24. Over-correcting/over-steering; 99. Unknown")
        fullNotUnknown = notNull.loc[notNull[i].isin(["99. Unknown", "06. Unknown if distracted", "07. Unknown if distracted"])==False] 
        fullNotUnknownPerc = np.nan_to_num(round(fullNotUnknown.shape[0]/df.shape[0]*100,0)).astype(int)
        filledNotUnknownPerc.append(fullNotUnknownPerc)
    
    Filled = {'Column':passengercolumns, 'Fill Percentage':filled, 'Non-Unknown Fill Percentage':filledNotUnknownPerc}
    Filled_PassengerTable = pd.DataFrame(Filled)
    Filled_PassengerTable.to_csv("Cleaned IMARS Passenger Table Fill Percentages_" + region+ ".csv", index=False)

### Vehicle Table

In [37]:
# add region column to vehicle table for parsing data into 7 reports
# note that this is the cleaned region data derived from Park name and will therefore keep as many reports as possible for each region
imars_vehicle_expanded = pd.merge(imars_vehicle, imars_crash_replaced[['RGN','IMARS_Record_No']], how ='left', on = 'IMARS_Record_No')
imars_vehicle_expanded.to_csv("imars_vehicle_rgncolumnadded.csv", index=False)

vehiclecolumns = imars_vehicle_expanded.columns
regions = ["NCR","NER","SER","MWR","IMR","PWR","AKR"]

for region in regions:
    df = imars_vehicle_expanded.loc[imars_vehicle_expanded['RGN']==region] 
    filled = []
    filledNotUnknownPerc = []
    for i in vehiclecolumns:
        notNull = df.loc[df[i].notnull()]
        # determine fill percentages (including "unknowns")
        full = np.nan_to_num(round(notNull.shape[0]/df.shape[0]*100,0)).astype(int)
        filled.append(full)

        # determine percent of reports where the crash reporter entered something OTHER THAN the unknown attributes listed 
            # (drop reports with these exact values, but keep reports where unknown was one of multiple REAL data attributes selected: e.g., "10. Lights (head, signal, tail); 99. Unknown")
        fullNotUnknown = notNull.loc[notNull[i].isin(["99. Unknown", "04. Unknown", "06. Unknown", "Unknown"])==False] 
        fullNotUnknownPerc = np.nan_to_num(round(fullNotUnknown.shape[0]/df.shape[0]*100,0)).astype(int)
        filledNotUnknownPerc.append(fullNotUnknownPerc)

    Filled = {'Column':vehiclecolumns, 'Fill Percentage':filled, 'Non-Unknown Fill Percentage':filledNotUnknownPerc}
    Filled_VehicleTable = pd.DataFrame(Filled)
    Filled_VehicleTable.to_csv("Cleaned IMARS Vehicle Table Fill Percentages_" + region + ".csv", index=False)